# DuckDB + SLayer, from the command line

The same demo as the [Python notebook](duckdb_python_nb.ipynb), driven entirely through the `slayer` command line. We install the DuckDB CLI, expose a **48 KB CSV on a CDN** as a DuckDB view, let SLayer **auto-ingest** its schema into a semantic model, and query it with `slayer query`. Nothing is copied locally — the view points at the URL and every query reaches back over the wire.

**Prerequisites:** `pip install motley-slayer`.

## 1. Set it up — all CLI

Three steps, no Python: install the DuckDB CLI, create a **view** over the remote CSV inside a DuckDB file, then point a SLayer datasource at that file and let `--ingest` **auto-build the model** by introspecting the view. Column names and types come straight from the data — nothing hand-written.

In [1]:
%%bash
set -euo pipefail
export SLAYER_STORAGE=.cache/cli/store
rm -rf .cache/cli && mkdir -p .cache/cli

# Install the DuckDB CLI if it isn't already on PATH (idempotent).
command -v duckdb >/dev/null 2>&1 || curl -fsSL https://install.duckdb.org | sh >/dev/null
export PATH="$HOME/.duckdb/cli/latest:$PATH"

# Expose the remote CSV as a DuckDB view: the CSV stays on the CDN, the view
# just points at it over httpfs — nothing is copied locally.
duckdb .cache/cli/weather.duckdb -c \
  "CREATE VIEW weather AS SELECT * FROM read_csv_auto('https://cdn.jsdelivr.net/npm/vega-datasets@2/data/seattle-weather.csv')"

# Register the datasource and auto-ingest the view into a semantic model —
# column names and types come from introspecting the view, nothing hand-written.
PYTHONWARNINGS=ignore slayer datasources create "duckdb:///$PWD/.cache/cli/weather.duckdb" \
  --name weather_db --ingest -y
slayer models list

Created datasource 'weather_db' (duckdb).
Ingested: weather (6 columns, 0 measures)


weather


## 2. A warm-up query

`slayer query` takes a JSON query. A plain grouped count — how many days of each weather type — rendered as a table.

In [2]:
%%bash
export SLAYER_STORAGE=.cache/cli/store
slayer query '{"source_model": "weather", "dimensions": ["weather"], "measures": [{"formula": "*:count", "name": "days"}]}' --format table

weather.weather | weather.days
--------------- | ------------
drizzle | 53
rain | 641
sun | 640
fog 

| 101
snow | 26

5 row(s)


## 3. The hero query: an aggregate as a dimension, plus year-over-year

One query, two recent features. Stage 1 totals monthly rainfall and its value twelve months back (`time_shift(..., -1, 'year')`, calendar-aware). Stage 2 bands each month *rainy* or *dry* by that total inside a `CASE WHEN` **dimension** — an aggregate used as a dimension — and regroups. A SLayer query is itself a model, so stage 2 just reads stage 1 by name. The first year (2012) has nothing to look back to, so its year-over-year values are null.

We write the query to a file and run it with `slayer query @file` — the same JSON the Python notebook hands to the client.

In [3]:
%%bash
export SLAYER_STORAGE=.cache/cli/store
cat > .cache/cli/hero.json <<'JSON'
[
  {
    "name": "monthly",
    "source_model": "weather",
    "time_dimensions": [{"dimension": "date", "granularity": "month"}],
    "measures": [
      {"formula": "precipitation:sum", "name": "rain"},
      {"formula": "precipitation:sum - time_shift(precipitation:sum, -1, 'year')", "name": "rain_yoy"}
    ]
  },
  {
    "source_model": "monthly",
    "dimensions": [
      {"expression": "CASE WHEN rain > 100 THEN 'rainy' ELSE 'dry' END", "name": "month_type"},
      "date"
    ],
    "measures": [
      {"formula": "rain:sum", "name": "total_rain"},
      {"formula": "rain_yoy:sum", "name": "total_rain_yoy"}
    ],
    "order": [{"column": "date", "direction": "asc"}]
  }
]
JSON
slayer query @.cache/cli/hero.json --format table

monthly.month_type | monthly.date | monthly.total_rain | monthly.total_rain_yoy
------------------ |

 ------------ | ------------------ | ----------------------
rainy | 2012-01-01 00:00:00 | 173.299999

99999998 | None
dry | 2012-02-01 00:00:00 | 92.3 | None
rainy | 2012-03-01 00:00:00 | 183.0 | None
d

ry | 2012-04-01 00:00:00 | 68.09999999999998 | None
dry | 2012-05-01 00:00:00 | 52.199999999999996 |

 None
dry | 2012-06-01 00:00:00 | 75.1 | None
dry | 2012-07-01 00:00:00 | 26.3 | None
dry | 2012-08-

...

-05-01 00:00:00 | 14.799999999999999 | -65.19999999999999
dry | 2015-06-01 00:00:00 | 5.899999999999

9995 | -12.900000000000006
dry | 2015-07-01 00:00:00 | 2.3 | -17.3
dry | 2015-08-01 00:00:00 | 83.3 

| 37.300000000000004
dry | 2015-09-01 00:00:00 | 21.1 | -35.599999999999994
rainy | 2015-10-01 00:00

:00 | 122.39999999999998 | -49.10000000000002
rainy | 2015-11-01 00:00:00 | 212.6 | 89.5
rainy | 201

5-12-01 00:00:00 | 284.5000000000001 | 162.70000000000013

48 row(s)


## 4. The SQL SLayer generated

`--dry-run` prints the single SQL statement for the whole two-stage query without executing it — the aggregate CTE, the year-shifted self-join, the banding and regroup on top. Exactly what DuckDB ran against the remote file.

In [4]:
%%bash
export SLAYER_STORAGE=.cache/cli/store
slayer query @.cache/cli/hero.json --dry-run

WITH monthly AS (
  SELECT
    _stage_inner."weather.date" AS "date",
    _stage_inner."weather.rain

" AS "rain",
    _stage_inner."weather.rain_yoy" AS "rain_yoy"
  FROM (
    SELECT
      "weather.da

te",
      "weather.rain",
      "weather.rain_yoy"
    FROM (
      WITH base AS (
        SELECT
 

         DATE_TRUNC('MONTH', weather.date) AS "weather.date",
          CAST(SUM(weather.precipitati

on) AS DOUBLE) AS "weather.rain"
        FROM weather AS weather
        GROUP BY
          DATE_TRU

...

her.rain",
        "weather._time_shift_inner",
        "weather.rain_yoy"
      FROM step1
    ) AS

 _outer
  ) AS _stage_inner
)
SELECT
  CASE WHEN monthly.rain > 100 THEN 'rainy' ELSE 'dry' END AS "

monthly.month_type",
  monthly.date AS "monthly.date",
  SUM(monthly.rain) AS "monthly.total_rain",


  SUM(monthly.rain_yoy) AS "monthly.total_rain_yoy"
FROM monthly AS monthly
GROUP BY
  CASE WHEN mon

thly.rain > 100 THEN 'rainy' ELSE 'dry' END,
  monthly.date
ORDER BY
  "monthly.date" ASC


---

The whole semantic layer over a file on the internet — a DuckDB view, auto-ingestion, and JSON queries, all from the shell. See the [Python notebook](duckdb_python_nb.ipynb) for the in-process library version, [multi-stage queries](../06_multistage_queries/multistage_queries.md) for the queries-as-models idea, and [formulas](../../concepts/formulas.md) for the full transform vocabulary.